# Match labelled rows with LLM-extracted rows
1. Load data
2. Vectorize rows
3. Match rows 
4. Compute accuracy


In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import geopy as gpy
import time
import itertools
import regex as re
from matplotlib import pyplot as plt
from src.data import *
from src.plot_functions import *
from src.post_process_functions import *
from src.geocoding import *
from src.hazard_def import *
from src.impact_def import *
from src.sanity_checks import *



## Load data

In [2]:
#load data (model)
post_processed = True #load or not post_processed data
model_name = "meta-llama/llama-4-scout-17b-16e-instruct"
nreports = 50
res_savename = f"post_processed_llm_response_impact_labelled_reports_{model_name.replace('/', '_')}.csv" if post_processed else f"llm_response_impact_labelled_reports_{model_name.replace('/', '_')}.csv"
extracted_df = pd.read_csv(DATA_OUT_PROC+res_savename)

#load data (labelled)
res_savename = "post_processed_labelled_reports_impacts_all.csv" if post_processed else "labelled_reports_impacts_all.csv"
labelled_df = pd.read_csv(DATA_OUT_PROC+res_savename)


In [2]:
#load geocoded data
post_processed = True #load or not post_processed data
model_name = "meta-llama/llama-4-scout-17b-16e-instruct"
nreports = 50
res_savename = f"post_processed_llm_response_impact_labelled_reports_{model_name.replace('/', '_')}_geo.gpkg"
extracted_df = gpd.read_file(DATA_OUT_PROC+res_savename)

#load data (labelled)
res_savename = "post_processed_labelled_reports_impacts_all_geo.gpkg"
labelled_df = gpd.read_file(DATA_OUT_PROC+res_savename)

In [7]:
#reformat output
num_cols = ["impactValue"]#"startYear", "startMonth", "startDay", "endYear", "endMonth", "endDay"
list_cols = ["country","location", "hazards", "impactsAnnotation"]
labelled_df = format_output(labelled_df, num_cols=num_cols, list_cols=list_cols)
extracted_df = format_output(extracted_df, num_cols=num_cols, list_cols=list_cols)
combined_df = pd.concat([labelled_df, extracted_df])


## Match
1. Vectorize columns that need to be compared using cosine similarity
2. Compute cosine similarity for those columns for each possible extracted-labelled pair
3. Add absolute difference of impactValue between each possible extracted-labelled pair.
    Need to consider NaN from not NaN separately. Only try matching non-NaNs with non-NaNs 
    (and nans with nan?)
4. Compute Intersection-Over-Union of polygons for each possible pair
5. Match by maximizing similarity and -impactvalu_idff and -IoT. Allow for more than one match.  


In [27]:
from sklearn.metrics.pairwise import cosine_similarity

def vectorize(cell_values, unique_values):
    """vectorizing function for categorical columns"""
    #cell_values = list() if not cell_values else cell_values
    cell_values = [cell_values] if not isinstance(cell_values, list) else cell_values
    vector = [1 if unique_value in cell_values else 0 for unique_value in unique_values]
    return np.array(vector)

def make_cosine_matrix(vec_df1, vec_df2, matching_cols):
    """Build cosine similarity matrix based on matching columns of vectorized dataframes"""
    sim_mat = np.full((len(vec_df1), len(vec_df2), len(matching_cols)), np.nan)

    for k, col in enumerate(matching_cols):
        # Convert Series of arrays/lists to 2D numpy arrays
        X = np.stack(vec_df1[col].values) #nsamples, nfeatures
        Y = np.stack(vec_df2[col].values)
        # Compute cosine similarity
        sim_mat[:,:,k] = cosine_similarity(X, Y)
    return sim_mat

def split_nans(df,key):
    """split a dataframe into two, according to whether the column key contains nans"""
    return df[~df[key].isna()], df[df[key].isna()]

def compute_iot(gdf_left, gdf_right):
    # Example: ensure both are in the same CRS
    gdf_left = gdf_left.to_crs(gdf_right.crs)

    # Precompute areas
    left_areas = gdf_left.geometry.area.values
    right_areas = gdf_right.geometry.area.values

    # Create empty matrix for IoUs
    iou_matrix = np.zeros((len(gdf_left), len(gdf_right)))

    # Compute pairwise IoUs
    for i, geom_left in enumerate(gdf_left.geometry):
        for j, geom_right in enumerate(gdf_right.geometry):
            intersection = geom_left.intersection(geom_right)
            if not intersection.is_empty:
                inter_area = intersection.area
                union_area = left_areas[i] + right_areas[j] - inter_area
                iou_matrix[i, j] = inter_area / union_area if union_area != 0 else 0

    return iou_matrix

In [38]:
geo_match = True
value_match = False

unique_countries_ISO = [country.alpha_3 for country in pycountry.countries]
unique_country_names = [country.name for country in pycountry.countries]
pattern = '|'.join(map(re.escape, unique_country_names))

unique_dict = {#mapping dictonary
    #'hazards' : hazard_main_types_emdat_extended,
    'country' : unique_country_names,
    'startYear' : np.arange(1980, 2025).tolist(),
    'startMonth' : np.arange(1, 13).tolist(),
    'startDay' : np.arange(1, 32).tolist(),
    'endYear' : np.arange(1980, 2025).tolist(),
    'endMonth' : np.arange(1, 13).tolist(),
    'endDay' : np.arange(1, 32).tolist(),
    'impactSubtype' : impactSubtype_list,
    'impactUnit' : combined_df.impactUnit.unique().tolist()
}

match_idx = []

for appeal, ext_group in extracted_df.groupby("appealCode"):
    matching_cols = list(unique_dict.keys())
    ext_group = ext_group.reset_index(drop=False, names=["orig_index"]) #need to reset index to get indices for numpy arrays
    lab_group = labelled_df[labelled_df["appealCode"] == appeal].reset_index(drop=False, names=["orig_index"])

    if lab_group.shape[0] == 0:
        continue

    ext_vect_df = pd.DataFrame(columns=matching_cols)
    lab_vect_df = pd.DataFrame(columns=matching_cols)

    #vectorize
    for col in matching_cols:
        ext_vect_df[col] = ext_group[col].apply(vectorize, unique_values=unique_dict[col])
        lab_vect_df[col] = lab_group[col].apply(vectorize, unique_values=unique_dict[col])

    #compute cosine distance
    dist_mat = make_cosine_matrix(ext_vect_df, lab_vect_df, matching_cols)

    ## TODO ADD IOTs
    if geo_match:
        #only compute ious on intersections to speed up
        #joined = gpd.sjoin(ext_group, lab_group, how="inner", predicate="intersects")
        #expand dist_mat to store results
        iou_mat = compute_iot(ext_group, lab_group)
        dist_mat = np.append(dist_mat, iou_mat[:,:, None], axis=2)

        #dist_mat = np.append(dist_mat, np.full((len(ext_vect_df), len(lab_vect_df), 1), np.nan), axis=2)
        #intersections = ext_group.overlay(lab_group, how="intersection", keep_geom_type=True).area
        #unions = ext_group.overlay(lab_group, how="union", keep_geom_type=True).area
        #dist_mat[:,:, -1] = intersections / unions
        #for irow, ext_row in ext_group.iterrows():
        #    #intersects each extracted rows with each labelled rows
        #    intersections = ext_row.overlay(lab_group[["geometry"]], how="intersection", keep_geom_type=True)
        #    unions = ext_row.overlay(lab_group[["geometry"]], how="union", keep_geom_type=True)
        #    dist_mat[irow, :, -1] = intersections.area / unions.area

    #need to separate Nans from not Nans
    #split
    not_nan_ext_df, nan_ext_df = split_nans(ext_group,"impactValue")
    not_nan_lab_df, nan_lab_df = split_nans(lab_group,"impactValue")

    #retrieve positional indices
    nan_id_ext = nan_ext_df.index.values
    nan_id_lab = nan_lab_df.index.values
    not_nan_id_ext = not_nan_ext_df.index.values
    not_nan_id_lab = not_nan_lab_df.index.values

    #remove nans from dist matrix
    if len(nan_id_ext):
        if len(nan_id_lab):
            dist_mat_notna = np.delete(np.delete(dist_mat, nan_id_ext, axis=0), nan_id_lab, axis=1)
            dist_mat_na = dist_mat[nan_id_ext, :, :][:,nan_id_lab,:]
        else:
            dist_mat_notna = np.delete(dist_mat, nan_id_ext, axis=0)
            dist_mat_na = dist_mat[nan_id_ext, :, :]
    else:
        if len(nan_id_lab):
            dist_mat_notna = np.delete(dist_mat, nan_id_lab, axis=1)
            dist_mat_na = None
        else:
            dist_mat_notna = dist_mat
            dist_mat_na = None

    #calculate diff
    if value_match:
        value_diff = 1-np.abs((not_nan_ext_df["impactValue"].values.reshape(-1, 1) - not_nan_lab_df["impactValue"].values.reshape(1, -1)) / not_nan_ext_df["impactValue"].values.reshape(-1, 1))
        dist_mat_notna = np.append(dist_mat_notna, value_diff[:,:, None], axis=2)

    #initialize
    id_match_ext = np.array([])
    id_match_lab = np.array([])

    #find possible candidates based on max similarity
    if dist_mat_notna.size != 0:
        agg_sim_notna = np.nansum(dist_mat_notna, axis=2) #aggregate score for all matching columns (#ext, #lab)
        max_sim_notna = np.max(agg_sim_notna, axis=1) #find max similarity value (#ext)
        id_match_ext_notna, id_match_lab_notna = np.where(agg_sim_notna == max_sim_notna[:,None]) #possible candidates with max similarity

        #match based on impact value

        #reindex back in original df
        id_match_ext_notna = not_nan_ext_df.iloc[id_match_ext_notna]["orig_index"].values.flatten()
        id_match_lab_notna = not_nan_lab_df.iloc[id_match_lab_notna]["orig_index"].values.flatten()
        id_match_ext = np.append(id_match_ext, id_match_ext_notna)
        id_match_lab = np.append(id_match_lab, id_match_lab_notna)

    #find possible candidates based on max similarity
    if dist_mat_na is not None:
        agg_sim_na = np.nansum(dist_mat_na, axis=2) #aggregate score for all matching columns (#ext, #lab)
        max_sim_na = np.max(agg_sim_na, axis=1) #find max similarity value (#ext)
        id_match_ext_na, id_match_lab_na = np.where(agg_sim_na == max_sim_na[:,None]) #possible candidates with max similarity
        id_match_ext_na = nan_ext_df.iloc[id_match_ext_na]["orig_index"].values.flatten()
        id_match_lab_na = nan_lab_df.iloc[id_match_lab_na]["orig_index"].values.flatten()
        id_match_ext = np.append(id_match_ext, id_match_ext_na)
        id_match_lab = np.append(id_match_lab, id_match_lab_na)
    #write as df
    match_idx.append(pd.DataFrame((id_match_ext, id_match_lab),
                               index = ["ext_match_id", "lab_match_id"]).T)
match_idx_df = pd.concat(match_idx)

/var/folders/y5/t1z41tgj7dv50sm2_29dn8740000gp/T/ipykernel_38457/2409377923.py:31: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  left_areas = gdf_left.geometry.area.values
/var/folders/y5/t1z41tgj7dv50sm2_29dn8740000gp/T/ipykernel_38457/2409377923.py:32: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  right_areas = gdf_right.geometry.area.values
/var/folders/y5/t1z41tgj7dv50sm2_29dn8740000gp/T/ipykernel_38457/2409377923.py:31: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  left_areas = gdf_left.geometry.area.values
/var/folders/y5/t1z41tgj7dv50sm2_29dn8740000gp/T/ipykernel_38457/240937

In [39]:
match_idx_df

,ext_match_id,lab_match_id
0,22.0,0.0
1,23.0,0.0
2,24.0,0.0
3,25.0,3.0
4,25.0,7.0
...,...,...
18,322.0,359.0
19,324.0,358.0
20,324.0,359.0
21,324.0,360.0


In [40]:
#join extracted and labelled dataframes
#repeated_df = labelled_df.loc[np.repeat(match_idx, 1)].reset_index(drop=True)
#repeated_df = repeated_df.add_suffix('_matched')
#joined_df = pd.concat([extracted_df, repeated_df], axis=1)

matched_df = pd.concat([extracted_df.loc[match_idx_df["ext_match_id"].values].reset_index(drop=True),
                        labelled_df.loc[match_idx_df["lab_match_id"].values].add_suffix('_matched').reset_index(drop=True)], axis=1)

In [41]:
matched_df.sort_index(axis=1)

,appealCode,appealCode_matched,comments_matched,country,country_iso3,country_iso3_kw,country_iso3_kw_matched,country_iso3_matched,country_kw,country_matched,...,reportDate_matched,reportLink,startDay,startDay_matched,startMonth,startMonth_matched,startYear,startYear_matched,unit_type,unit_type_matched
0,MDRBD022,MDRBD022,NaN,[Bangladesh],NaN,BGD,NaN,NaN,['Bangladesh'],[Bangladesh],...,2019-07-19,https://adore.ifrc.org/Download.aspx?FileId=36...,NaN,18.0,7,7.0,2019.0,2019.0,other,other
1,MDRBD022,MDRBD022,NaN,[Bangladesh],NaN,BGD,NaN,NaN,['Bangladesh'],[Bangladesh],...,2019-07-19,https://adore.ifrc.org/Download.aspx?FileId=36...,NaN,18.0,7,7.0,2019.0,2019.0,other,other
2,MDRBD022,MDRBD022,NaN,[Bangladesh],NaN,BGD,NaN,NaN,['Bangladesh'],[Bangladesh],...,2019-07-19,https://adore.ifrc.org/Download.aspx?FileId=36...,NaN,18.0,7,7.0,2019.0,2019.0,other,other
3,MDRBD022,MDRBD022,NaN,[Bangladesh],NaN,BGD,NaN,NaN,['Bangladesh'],[Bangladesh],...,2019-07-19,https://adore.ifrc.org/Download.aspx?FileId=36...,NaN,16.0,7,7.0,2019.0,2019.0,other,other
4,MDRBD022,MDRBD022,NaN,[Bangladesh],NaN,BGD,NaN,NaN,['Bangladesh'],[Bangladesh],...,2019-07-19,https://adore.ifrc.org/Download.aspx?FileId=36...,NaN,16.0,7,7.0,2019.0,2019.0,other,other
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
386,MDRZM022,MDRZM022,NaN,[Zambia],NaN,ZMB,NaN,NaN,['Zambia'],[Zambia],...,2024-04-02,https://adore.ifrc.org/Download.aspx?FileId=84...,NaN,29.0,NaN,2.0,2023.0,2024.0,other,other
387,MDRZM022,MDRZM022,NaN,[Zambia],NaN,ZMB,NaN,NaN,['Zambia'],[Zambia],...,2024-04-02,https://adore.ifrc.org/Download.aspx?FileId=84...,NaN,29.0,NaN,2.0,NaN,2024.0,other,other
388,MDRZM022,MDRZM022,NaN,[Zambia],NaN,ZMB,NaN,NaN,['Zambia'],[Zambia],...,2024-04-02,https://adore.ifrc.org/Download.aspx?FileId=84...,NaN,29.0,NaN,2.0,NaN,2024.0,other,other
389,MDRZM022,MDRZM022,NaN,[Zambia],NaN,ZMB,NaN,NaN,['Zambia'],[Zambia],...,2024-04-02,https://adore.ifrc.org/Download.aspx?FileId=84...,NaN,29.0,NaN,2.0,NaN,2024.0,other,other


In [42]:
matched_df[["appealCode", "impactSubtype", "impactSubtype_matched" ,"impactValue", "impactUnit", "impactValue_matched", "impactUnit_matched", "impactsAnnotation", "impactsAnnotation_matched"]]

,appealCode,impactSubtype,impactSubtype_matched,impactValue,impactUnit,impactValue_matched,impactUnit_matched,impactsAnnotation,impactsAnnotation_matched
0,MDRBD022,Affected People,Affected People,7600000.0,people,2176519.0,people,[more than 7.6 million people were affected in...,[DREF operation n MDRBD022 Glide n FL-2019-000...
1,MDRBD022,Displaced People,Affected People,300000.0,people,2176519.0,people,"[over 300,000 people displaced]",[DREF operation n MDRBD022 Glide n FL-2019-000...
2,MDRBD022,Human Deaths,Affected People,114.0,people,2176519.0,people,[114 people dead],[DREF operation n MDRBD022 Glide n FL-2019-000...
3,MDRBD022,Residential Buildings,Residential Buildings,600000.0,homes,10000.0,homes,"[approximately 600,000 houses damaged]",[According to National disaster response coord...
4,MDRBD022,Residential Buildings,Residential Buildings,600000.0,homes,3988.0,homes,"[approximately 600,000 houses damaged]","[of fully damaged house 3,988 No., Six days of..."
...,...,...,...,...,...,...,...,...,...
386,MDRZM022,Affected Livestock and Animals,Affected Livestock and Animals,NaN,undefined affected animals,NaN,undefined affected animals,[almost half of surveyed households that kept ...,[Page 1 / 21 DREF Operation Zambia Drought 202...
387,MDRZM022,Access to Food,Water Quality and Availability,NaN,people,NaN,unknown,[decreased access to water has also led to out...,[Page 1 / 21 DREF Operation Zambia Drought 202...
388,MDRZM022,Access to Food,Affected Livestock and Animals,NaN,people,NaN,undefined affected animals,[decreased access to water has also led to out...,[Page 1 / 21 DREF Operation Zambia Drought 202...
389,MDRZM022,Access to Food,Other Economic and Livelihood Impacts,NaN,people,NaN,CHF,[decreased access to water has also led to out...,[Page 1 / 21 DREF Operation Zambia Drought 202...
